# HUM-450 Preliminary Analysis
The goal of this notebooks is to develop and test analysis tool for the project

In [ ]:
import numpy as np
import pandas as pd
import os
import re
from collections import Counter, defaultdict
import networkx as nx
import folium
from folium.plugins import MarkerCluster
import geopy

#Change the root path
ROOT="/Users/edouardkoehn/Documents/Github/HUM-450"
path_data=ROOT+"/data"

## Location Analysis

In [ ]:
def load_database(path):
    """
    Simple method for loading the database
    """
    database = pd.read_csv(path, sep=';')
    database['persons_mentioned']=database['persons_mentioned'].apply(lambda x: x[1:-1].replace("'", '').replace(", ",",").split(','))
    database['locations_mentioned']=database['locations_mentioned'].apply(lambda x: x[1:-1].replace("'", '').replace(", ",",").split(','))
    return database

def get_location_dist(data):
    """
    Compute the distribution of locations mentioned in the data.

    Parameters:
    - data (pd.DataFrame): DataFrame containing the column 'locations_mentioned' listing the locations mentioned.

    Returns:
    - ind (defaultdict): Dictionary containing the location distribution, where each location is associated with its probability.
    """
    temp = []
    for locations in data['locations_mentioned']:
        temp = np.concatenate([temp, locations])
        ind = Counter(temp)
        size_index = [word_count for word, word_count in ind.items()]
        size_index = np.sum(size_index)
        ind = defaultdict(float, {word: word_count/size_index for word, word_count in ind.items()})
        
    return ind

def top_k_location(indexes, k):
    """
    Return the top k locations from the given location distribution.

    Parameters:
    - indexes (dict): Dictionary containing the location distribution.
    - k (int): Number of top locations to return.

    Returns:
    - top_k (list): List of tuples containing the top k locations and their probabilities.
    """
    return sorted(indexes.items(), key=lambda item: item[1], reverse=True)[:k]

def get_map_location(location_distribution, k):
    """
    Generate a folium map showing the distribution of top k locations.

    Parameters:
    - location_distribution (dict): Dictionary containing the location distribution.
    - k (int): Number of top locations to display on the map.

    Returns:
    - lausanne_map (folium.Map): Folium map object displaying the distribution of top k locations.
    """
    geocoder = geopy.Nominatim(user_agent="tutorial")
    top_k_destination = top_k_location(location_distribution, k)

    coordinates = []
    for i in top_k_destination:
        coordinates.append(geocoder.geocode(i[0]))
    
    lausanne_map = folium.Map(location=[46.53, 6.64], zoom_start=13, tiles='CartoDB positron', control_scale=True)
    for top, coord in zip(top_k_destination, coordinates):
        folium.Circle(
            radius=top[1]*100000,
            location=coord[1]
        ).add_to(lausanne_map)
    return lausanne_map

In [ ]:
#Load the data
extraction=load_database(path_data+"/database_1890-1910.csv")
extraction.head()

In [ ]:
locations_dist=get_location_dist(extraction)
top_k_location(locations_dist, 10)

In [ ]:
get_map_location(locations_dist,20)